[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rehim2009book-ui/fairloan/blob/main/fairloan.ipynb)

# TwinCheck: catching hidden age bias in loan decisions

**Veritas AI International High School Hackathon 2026**

**Track:** AI + Finance (built on suggested project 4.2.3, Fairness audit of a loan model)
**Team:** FairLoan

**One-sentence pitch:** TwinCheck helps a young person who was refused a loan by taking their application and giving back the decision, the real reasons in plain words, what they could change to get a yes, and a check of whether the bank's model treats young people worse even though it is never told their age.

**How to run:** Runtime → Run all. It takes about a minute and needs no API key or uploads. The AI-written explanations are loaded from a saved file in our repository.

*This project is a prototype and is not financial advice.*

### Why we built this

Banks use computer models to decide who gets a loan. A model can be unfair without anyone noticing. In the Netherlands, the tax office used an algorithm that treated nationality as a risk sign. Tens of thousands of families were wrongly accused of fraud, and the Dutch government resigned over it in January 2021.

The usual fix is "just don't give the model the sensitive detail". We test whether that fix works for age, and we find it does not.

### What is ours (beyond the brief)

The brief asks for a loan model and the approval-rate gap between two age groups. We added:

1. **Twin Test.** Two applicants who are identical except for one detail. How often does the decision flip?
2. **Age guesser.** A second model that tries to guess who is under 25 without seeing age.
3. **Fair test.** Among people who really paid back, who gets wrongly rejected more often?
4. **Two fixes judged on money**, using the dataset's own cost rule.
5. **"What would get a yes"** for each rejected person.
6. **AI explanations, and a lie check on them.**
7. **Corrected labels.** The dataset's official code table is wrong for 8 columns. We found this and used the corrected table.

## 1. Load the data

We use the **Statlog German Credit** data (UCI Machine Learning Repository): 1,000 people who took a loan from a German bank in the 1970s, 20 details about each, and whether they turned out to be a good or bad risk.

**A trap we fell into and fixed.** The code table published with this dataset is wrong for 8 columns. For example, the code it calls "rent" really means "lives rent-free", and the code it calls "no property" really means "real estate". Ulrike Grömping showed this in 2019. Our first run used the wrong table, and our AI explanations told an applicant the opposite of the truth. We now use the corrected table.

We do not use two columns at all: *personal status and sex* (sex cannot be recovered from it) and *foreign worker* (its coding is unreliable).

In [ ]:
!pip install -q ucimlrepo

In [ ]:
# Load data and make it readable
# What it does: downloads the 1,000 loan applicants, renames the cryptic columns, turns codes into words.
import pandas as pd, numpy as np
from ucimlrepo import fetch_ucirepo

raw = fetch_ucirepo(id=144)
X = raw.data.features.copy()
y_raw = raw.data.targets["class"]          # 1 = good (paid back), 2 = bad

names = {
 "Attribute1":"checking","Attribute2":"duration_months","Attribute3":"credit_history","Attribute4":"purpose",
 "Attribute5":"amount","Attribute6":"savings","Attribute7":"employment","Attribute8":"installment_rate",
 "Attribute9":"personal_status","Attribute10":"other_debtors","Attribute11":"residence_years","Attribute12":"property",
 "Attribute13":"age","Attribute14":"other_plans","Attribute15":"housing","Attribute16":"existing_credits",
 "Attribute17":"job","Attribute18":"people_liable","Attribute19":"telephone","Attribute20":"foreign_worker"}
X = X.rename(columns=names)

# IMPORTANT: the code table printed in the original UCI documentation is wrong for several columns.
# We use the corrected meanings from Groemping (2019), "South German Credit Data: Correcting a Widely Used Data Set".
codes = {
 "checking":{"A11":"no account","A12":"<0","A13":"0-200","A14":">=200 or salary"},
 "credit_history":{"A30":"delays in the past","A31":"critical / credits elsewhere","A32":"no credits / all paid back",
                   "A33":"existing credits paid so far","A34":"all credits here paid back"},
 "purpose":{"A40":"other","A41":"new car","A42":"used car","A43":"furniture","A44":"radio/tv","A45":"appliances",
            "A46":"repairs","A47":"education","A48":"vacation","A49":"retraining","A410":"business"},
 "savings":{"A61":"none/unknown","A62":"<100","A63":"100-500","A64":"500-1000","A65":">=1000"},
 "employment":{"A71":"unemployed","A72":"<1yr","A73":"1-4yr","A74":"4-7yr","A75":">=7yr"},
 "other_debtors":{"A101":"none","A102":"co-applicant","A103":"guarantor"},
 "property":{"A121":"none","A122":"car/other","A123":"life insurance or building plan","A124":"real estate"},
 "other_plans":{"A141":"bank","A142":"stores","A143":"none"},
 "housing":{"A151":"rent-free","A152":"rent","A153":"own"},  # "rent-free" = lives somewhere without paying rent, e.g. with family
 "job":{"A171":"unskilled non-resident","A172":"unskilled","A173":"skilled","A174":"management"},
 "telephone":{"A191":"no","A192":"yes"},
 # personal_status (sex cannot be recovered) and foreign_worker (coding unreliable) are left as codes and NOT used.
}
for col, m in codes.items():
    X[col] = X[col].map(m).fillna(X[col])

df = X.copy()
df["approve"] = (y_raw == 1).astype(int)            # 1 = good risk = approve
df["young"] = (df["age"] < 25).astype(int)           # the group we audit
print(df.shape)
df.head()

## 2. Look at the data first

Before training anything we count people and compare the two age groups: under 25, and 25 or older.

In [ ]:
# Look at the data
# What it does: counts people, approvals, and how many are under 25.
print("Applicants:", len(df))
print("Good risk (approve):", df.approve.sum(), "=", f"{df.approve.mean():.0%}")
print("Under 25:", df.young.sum(), " | 25 and over:", (1-df.young).sum())
print("\nReal repayment rate by age group:")
print(df.groupby(df.young.map({1:"under 25",0:"25+"}))["approve"].mean().round(3))
print("\nHousing by age group (share of each group):")
print(pd.crosstab(df.young.map({1:"under 25",0:"25+"}), df["housing"], normalize="index").round(2))

**What we see.** Young people in this data really do pay back less often (59.1% against 71.9%). So some gap between the groups is expected, and we have to be careful about what counts as unfair.

Also: 46% of under-25s live rent-free (for example with family), against 13% of older applicants. That detail can give age away.

## 3. Baseline: approve everyone

The simplest possible rule is to say yes to everyone. Any model has to beat it.

We score with the dataset's own cost rule: **approving a bad risk costs 5, rejecting a good one costs 1.** Lower cost is better. We keep 20% of the people hidden for testing.

In [ ]:
# Split + baseline + cost
# What it does: holds back 20% for testing. Baseline = approve everyone. Cost = official 5:1 rule.
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

FEATURES = [c for c in df.columns if c not in ["approve","young","foreign_worker","personal_status"]]
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df["approve"], test_size=0.2, stratify=df["approve"], random_state=42)
young_test = (X_test["age"] < 25).astype(int)

def cost(y_true, y_pred):
    """Bank's money lost. Approving a bad risk costs 5, rejecting a good one costs 1."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    bad_approved = ((y_true==0)&(y_pred==1)).sum()
    good_rejected = ((y_true==1)&(y_pred==0)).sum()
    return 5*bad_approved + 1*good_rejected

def report(name, y_true, y_pred):
    print(f"{name}")
    print(f"  accuracy          {accuracy_score(y_true,y_pred):.1%}")
    print(f"  bad-risk recall   {recall_score(1-np.asarray(y_true),1-np.asarray(y_pred)):.1%}   (share of bad risks we caught)")
    print(f"  cost (5:1 rule)   {cost(y_true,y_pred)}   (lower is better)")

baseline = np.ones(len(y_test), dtype=int)
report("Baseline: approve everyone", y_test, baseline)

## 4. Our model

We train two standard models, a logistic regression and a random forest, and keep the one that costs the bank less on the hidden test people.

In [ ]:
# Train two models, pick one
# What it does: logistic regression + random forest. We keep whichever loses less money.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

num_cols = ["duration_months","amount","installment_rate","residence_years","age","existing_credits","people_liable"]
cat_cols = [c for c in FEATURES if c not in num_cols]

def make_pipe(model, features=FEATURES):
    nums = [c for c in num_cols if c in features]; cats = [c for c in cat_cols if c in features]
    pre = ColumnTransformer([("num", StandardScaler(), nums),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cats)])
    return Pipeline([("pre", pre), ("model", model)])

lr = make_pipe(LogisticRegression(max_iter=2000, class_weight="balanced")).fit(X_train, y_train)
rf = make_pipe(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")).fit(X_train, y_train)

report("\nLogistic regression", y_test, lr.predict(X_test))
report("\nRandom forest",       y_test, rf.predict(X_test))

model = lr if cost(y_test, lr.predict(X_test)) <= cost(y_test, rf.predict(X_test)) else rf
print("\nWe keep:", "logistic regression" if model is lr else "random forest")

**What we see.** By accuracy, our model (67.0%) looks *worse* than approving everyone (70.0%). By money it is about twice as good (cost 142 against 300), because it catches most of the bad risks. Accuracy is the wrong measure here.

## 5. Audit 1: does the model approve young people less often?

This is the check the brief asks for: the approval rate for each age group, and the gap.

In [ ]:
# Audit I - does it approve young people less?
# What it does: approval rate for under-25 vs 25+, on the test set. The gap is our headline number.
pred = model.predict(X_test)
tab = pd.DataFrame({"approved": pred, "group": young_test.map({1:"under 25",0:"25+"})})
rates = tab.groupby("group")["approved"].agg(["mean","count"]).rename(columns={"mean":"approval_rate","count":"people"})
print(rates.round(3))
gap = rates.loc["25+","approval_rate"] - rates.loc["under 25","approval_rate"]
print(f"\nGap: 25+ approved {gap:.1%} more often than under-25s.")
print("(Note: the real repayment rates also differ by age - see cell 2. The question is whether the model's gap is bigger than the data justifies, and what drives it. Cells 6, 7 and 7b answer that.)")

## 6. Audit 2: delete age. Is the gap gone?

The usual answer to age bias is to remove the age column. We do that and measure again.

Only 28 of the 200 test people are under 25, which is too few to trust. So we also score all 1,000 people with *cross-validation*: each person is judged by a model that was trained without them. That gives 149 under-25s.

Then we build an **age guesser**: a second model that tries to tell who is under 25 from the other details, without seeing age.

In [ ]:
# Audit II - delete age. Is it still unfair? And can the model guess age anyway?
# What it does: (a) retrain WITHOUT the age column, measure the gap again (on the test set AND on all 1,000 via cross-validation).
#               (b) "attacker": a model that tries to guess who is under 25 from the other columns.
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

FEATURES_NOAGE = [c for c in FEATURES if c != "age"]
model_noage = make_pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), FEATURES_NOAGE).fit(X_train[FEATURES_NOAGE], y_train)

def approval_gap(pred, young_flag):
    r = pd.Series(pred).groupby(np.asarray(young_flag)).mean()
    return r[0] - r[1]            # 25+ rate minus under-25 rate

pred_noage = model_noage.predict(X_test[FEATURES_NOAGE])
gap_noage = approval_gap(pred_noage, young_test)
print("TEST SET (200 people, only 28 under 25 - noisy):")
print(f"  gap WITH age column:    {gap:.1%}")
print(f"  gap WITHOUT age column: {gap_noage:.1%}")
print(f"  cost without age: {cost(y_test, pred_noage)}  (with age: {cost(y_test, pred)})")

# steadier: predict every one of the 1,000 people with cross-validation (each person predicted by a model that never saw them)
cv_with    = cross_val_predict(make_pipe(LogisticRegression(max_iter=2000, class_weight="balanced")), df[FEATURES], df["approve"], cv=5)
cv_without = cross_val_predict(make_pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), FEATURES_NOAGE), df[FEATURES_NOAGE], df["approve"], cv=5)
gap_cv_with, gap_cv_without = approval_gap(cv_with, df["young"]), approval_gap(cv_without, df["young"])
print("\nALL 1,000 PEOPLE (149 under 25, cross-validated):")
print(f"  gap WITH age column:    {gap_cv_with:.1%}")
print(f"  gap WITHOUT age column: {gap_cv_without:.1%}")
if gap_cv_without > 0.5*gap_cv_with:
    print("  -> deleting age removes less than half the gap. The model finds age another way.")
elif gap_cv_without > 0.02:
    print("  -> deleting age shrinks the gap, but a real gap remains. The model finds age another way.")
else:
    print("  -> deleting age closes most of the gap in this data.")

# (b) attacker: guess 'young' from everything except age
attacker = make_pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), FEATURES_NOAGE)
guess = cross_val_predict(attacker, df[FEATURES_NOAGE], df["young"], cv=5, method="predict_proba")[:,1]
auc = roc_auc_score(df["young"], guess)
print(f"\nAttacker AUC for guessing 'under 25' without seeing age: {auc:.2f}   (0.50 = coin flip, 1.00 = perfect)")

attacker.fit(df[FEATURES_NOAGE], df["young"])
feat_names = attacker.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(attacker.named_steps["model"].coef_[0], index=feat_names)
print("\nTop clues that someone is YOUNG:");  print(coefs.sort_values(ascending=False).head(6).round(2))
print("\nTop clues that someone is OLDER:"); print(coefs.sort_values().head(6).round(2))

**What we see.** Across all 1,000 people the gap only shrinks from 17.4 to 13.8 points when age is deleted. The age guesser scores 0.78 (0.50 would be a coin flip, 1.00 perfect), and its strongest clue is living rent-free. So deleting the age column does not hide age from the model.

## 7. Audit 3: the Twin Test (our main addition)

Researchers test for discrimination in hiring by sending out two CVs that are identical except for one detail. We do the same to the model.

For each detail, every applicant gets a twin who is identical except for that one detail. One twin gets the value most typical of under-25s, the other the value most typical of older people. **The data picks those values, not us.** Then we count how often the model's answer flips.

We split details into two kinds. *Money details* (savings, checking account) are things a bank may fairly look at. *Stand-ins for age* are details like housing that mostly tell you how old someone is.

In [ ]:
# Audit III - the Twin Test
# What it does: for each detail, every applicant gets an identical twin that differs ONLY in that detail.
# One twin gets the value most typical of under-25s, the other the value most typical of over-25s.
# (The data picks those values, not us.) We count how often the age-free model changes its answer.
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

is_young = df["young"].values.astype(bool)

def typical_pair(feature):
    """Value most over-represented among under-25s, and value most over-represented among 25+."""
    sy = df.loc[is_young, feature].value_counts(normalize=True)
    so = df.loc[~is_young, feature].value_counts(normalize=True)
    idx = sy.index.union(so.index)
    diff = sy.reindex(idx, fill_value=0) - so.reindex(idx, fill_value=0)
    return diff.idxmax(), diff.idxmin()

STAND_INS = ["housing", "employment", "job", "telephone", "property", "residence_years"]   # details that can leak age
MONEY = [("savings", "<100", ">=1000"), ("checking", "<0", ">=200 or salary")]              # details a bank may fairly use

pairs = [(f, *typical_pair(f), "stand-in for age") for f in STAND_INS] + [(f, a, b, "money") for f, a, b in MONEY]

rows = []
for f, yv, ov, kind in pairs:
    a = df[FEATURES_NOAGE].copy(); a[f] = yv          # young-typical twin
    b = df[FEATURES_NOAGE].copy(); b[f] = ov          # older-typical twin
    pa, pb = model_noage.predict(a), model_noage.predict(b)
    flips = pa != pb
    young_twin_loses = ((pa == 0) & (pb == 1)).mean(); old_twin_loses = ((pa == 1) & (pb == 0)).mean()
    rows.append({"feature": f, "young-typical vs older-typical": f"{yv} vs {ov}", "kind": kind,
                 "flips (everyone)": flips.mean(), "flips (real under-25s)": flips[is_young].mean(),
                 "who loses": ("first twin" if young_twin_loses >= old_twin_loses else "second twin") if kind == "money"
                              else ("young-typical twin" if young_twin_loses >= old_twin_loses else "older-typical twin")})
twins = pd.DataFrame(rows).sort_values("flips (everyone)", ascending=False).reset_index(drop=True)
print(twins.to_string(index=False, formatters={"flips (everyone)":"{:.0%}".format, "flips (real under-25s)":"{:.0%}".format}))

def bar_color(r):
    if r["kind"] == "money": return "#999999"
    return "#c0392b" if r["who loses"] == "young-typical twin" else "#e8b04a"
plt.figure(figsize=(9,4.8))
plt.barh(twins["feature"] + "\n(" + twins["young-typical vs older-typical"] + ")", twins["flips (everyone)"]*100,
         color=[bar_color(r) for _, r in twins.iterrows()])
plt.gca().invert_yaxis()
plt.xlabel("% of applicants whose decision FLIPS when only this one detail changes")
plt.title("Twin Test - the model never sees age")
plt.legend(handles=[Patch(color="#c0392b", label="leaks age, young-typical twin loses"),
                    Patch(color="#e8b04a", label="leaks age, older-typical twin loses"),
                    Patch(color="#999999", label="money detail")], loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

hurts = twins[(twins.kind == "stand-in for age") & (twins["who loses"] == "young-typical twin")]
if len(hurts):
    top = hurts.iloc[0]
    print(f"\nStrongest age stand-in that works AGAINST the young-typical twin: {top['feature']} ({top['young-typical vs older-typical']})"
          f" - flips the decision for {top['flips (everyone)']:.0%} of applicants. The model was never told anyone's age.")
    # One concrete twin for the demo: a real under-25 whose decision flips when only that detail changes
    f = top["feature"]; yv, ov = [p for p in pairs if p[0] == f][0][1:3]
    demo_feature, demo_young_val, demo_old_val = f, yv, ov          # remembered for the demo cell
    cands = df[is_young & (df[f] == yv)][FEATURES_NOAGE]
    cands_twin = cands.copy(); cands_twin[f] = ov
    flipped = (model_noage.predict(cands) == 0) & (model_noage.predict(cands_twin) == 1)
    leyla = cands[flipped].iloc[[0]] if flipped.any() else cands.iloc[[0]]
    twin  = leyla.copy(); twin[f] = ov
    p1 = model_noage.predict_proba(leyla)[0,1]; p2 = model_noage.predict_proba(twin)[0,1]
    print(f"\nLeyla, applicant #{leyla.index[0]} ({f}: {yv}):  approve chance {p1:.0%}  -> {'APPROVED' if p1>=0.5 else 'REJECTED'}")
    print(f"Her twin ({f}: {ov}):  approve chance {p2:.0%}  -> {'APPROVED' if p2>=0.5 else 'REJECTED'}")
    print("Everything else about them is identical. The model was never told their age.")
    print(f"\nReal under-25s with {f} = {yv} who are rejected, but would be approved with {f} = {ov}: {flipped.sum()} of {len(cands)}")
else:
    leyla = df[is_young & (model_noage.predict(df[FEATURES_NOAGE]) == 0)].iloc[[0]][FEATURES_NOAGE]
    print("\nNo age stand-in works against the young-typical twin in this model.")

**What we see.** Money details matter most, which is how it should be. But housing alone (rent-free against renting) flips the decision for 13% of applicants, and 7 real under-25s who live rent-free are rejected although they would be approved as renters. The model was never told anyone's age.

The effect is real but modest. One odd result: for property, it is the older-typical twin who loses.

## 8. Audit 4: the fair test

Young people in this data pay back less often, so a gap in approvals is not automatically unfair. A fairer question: **look only at people who really did pay back. Does the model wrongly reject the young ones more often?**

In [ ]:
# The fair question - among people who really PAID BACK, who got wrongly rejected?
# What it does: young people in this data repay a bit less often, so some gap is expected.
# The fair test is: look ONLY at people who did pay back. Does the model wrongly reject the young ones more often?
good = df["approve"].values == 1
print(f"People who really paid back: {(good & is_young).sum()} under 25, {(good & ~is_young).sum()} aged 25+\n")
for label, p in [("model WITH age", cv_with), ("model WITHOUT age", cv_without)]:
    wrong_young = (p[good & is_young] == 0).mean()
    wrong_old   = (p[good & ~is_young] == 0).mean()
    print(f"{label}:")
    print(f"   good payers wrongly rejected, under 25: {wrong_young:.1%}")
    print(f"   good payers wrongly rejected, 25+:      {wrong_old:.1%}")
    print(f"   -> a young good payer is {wrong_young/wrong_old:.1f}x as likely to be wrongly rejected\n")
print(f"For comparison, real repayment rates: under 25 {df.approve[is_young].mean():.1%}, 25+ {df.approve[~is_young].mean():.1%}")

**What we see.** Yes. Without the age column, 37.5% of young good payers are wrongly rejected against 26.8% of older good payers, about 1.4 times as often. There are only 88 young good payers, so we call this a strong sign and not proof.

## 9. Two fixes, judged on money

- **Fix A:** also delete the stand-in details the Twin Test found.
- **Fix B:** keep the model, but give under-25s a slightly lower pass mark, set so that good payers in both age groups are wrongly rejected about equally often. The pass mark is chosen on other people's data.

We score every version on cost (per 1,000 applicants) and on fairness.

In [ ]:
# The fix - make the bot fairer, and show what it costs the bank
# What it does: tries two simple fixes and scores every version on money (5:1 rule) and on fairness.
#   Fix A: also delete the details that leak age and work against young people (found by the Twin Test).
#   Fix B: keep the model, but use a slightly lower pass mark for under-25s, so that good payers of both
#          age groups are wrongly rejected about equally often. The pass mark is picked on other people's data.
# All numbers are for all 1,000 applicants, each one scored by a model that never saw them (cross-validation).
from sklearn.model_selection import StratifiedKFold

y_all = df["approve"].values
def new_lr(): return LogisticRegression(max_iter=2000, class_weight="balanced")

proba_noage = cross_val_predict(make_pipe(new_lr(), FEATURES_NOAGE), df[FEATURES_NOAGE], df["approve"], cv=5, method="predict_proba")[:, 1]

# Fix A: drop the age stand-ins that hurt the young-typical twin (flip rate of 5% or more)
DROP = hurts.loc[hurts["flips (everyone)"] >= 0.05, "feature"].tolist()
FEATURES_A = [c for c in FEATURES_NOAGE if c not in DROP]
pred_A = cross_val_predict(make_pipe(new_lr(), FEATURES_A), df[FEATURES_A], df["approve"], cv=5)
print("Fix A deletes these details:", DROP if DROP else "(none found)")

# Fix B: lower pass mark for under-25s
def young_pass_mark(p, y, young):
    target = (p[(y == 1) & ~young] < 0.5).mean()                    # how often older good payers are wrongly rejected
    marks = np.arange(0.20, 0.505, 0.01)
    rates = np.array([(p[(y == 1) & young] < m).mean() for m in marks])
    return float(marks[np.argmin(np.abs(rates - target))])

pred_B = (proba_noage >= 0.5).astype(int); marks_used = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(proba_noage, y_all):
    m = young_pass_mark(proba_noage[tr], y_all[tr], is_young[tr]); marks_used.append(m)
    te_young = te[is_young[te]]
    pred_B[te_young] = (proba_noage[te_young] >= m).astype(int)
print(f"Fix B pass mark for under-25s: about {np.mean(marks_used):.2f} (everyone else: 0.50)\n")

def summary(name, p):
    p = np.asarray(p)
    return {"version": name, "cost": cost(y_all, p), "accuracy": (p == y_all).mean(),
            "approval gap": approval_gap(p, df["young"]),
            "good payers wrongly rejected, under 25": (p[(y_all == 1) & is_young] == 0).mean(),
            "same, 25+": (p[(y_all == 1) & ~is_young] == 0).mean()}

results = pd.DataFrame([
    summary("approve everyone", np.ones(len(df), dtype=int)),
    summary("model with age", cv_with),
    summary("model without age", cv_without),
    summary("Fix A: also delete age stand-ins", pred_A),
    summary("Fix B: lower pass mark under 25", pred_B),
])
pct = "{:.1%}".format
print(results.to_string(index=False, formatters={"accuracy": pct, "approval gap": pct,
      "good payers wrongly rejected, under 25": pct, "same, 25+": pct}))

base = results.iloc[2]
for i in [3, 4]:
    r = results.iloc[i]
    print(f"\n{r['version']}: cost {base['cost']} -> {r['cost']} ({r['cost']-base['cost']:+d}), "
          f"approval gap {base['approval gap']:.1%} -> {r['approval gap']:.1%}, "
          f"young good payers wrongly rejected {base['good payers wrongly rejected, under 25']:.1%} -> {r['good payers wrongly rejected, under 25']:.1%}")
print("\n(Cost: approving a bad risk = 5, rejecting a good one = 1. Lower is better. 'approve everyone' is the baseline.)")

**What we see.** There is no free fix.

- Fix A costs the bank nothing but barely helps young good payers (37.5% → 36.4%).
- Fix B makes wrongful rejections equal (27.3% against 26.8%), but costs the bank about 15% more (617 → 708). It also uses age on purpose, and under-25s then get approved slightly more often than older people.

Every version is still far cheaper than approving everyone (1,500).

## 10. What would get a yes?

For a rejected person we try small realistic changes (ask for less money, pay back faster, build savings, improve the checking account, add a guarantor, stay longer in the job) and report the smallest one that turns the answer into a yes. It never suggests "be older".

This shows what would flip *this model*. It is not a promise from a bank.

In [ ]:
# What would get me a yes?
# What it does: for a rejected person, tries small realistic changes and reports the smallest one that turns the
# answer into a yes. It never suggests "be older". It shows what flips THIS model - it is not a promise from a bank.
from itertools import combinations

SAVINGS_ORDER  = ["none/unknown", "<100", "100-500", "500-1000", ">=1000"]
CHECKING_ORDER = ["<0", "0-200", ">=200 or salary"]
EMPLOY_ORDER   = ["unemployed", "<1yr", "1-4yr", "4-7yr", ">=7yr"]

def possible_changes(person):
    """List of (feature, new value, plain text, effort). Lower effort = smaller change."""
    r = person.iloc[0]; out = []
    for k, cut in enumerate([10, 20, 30, 40, 50], start=1):
        new = int(round(r["amount"] * (1 - cut / 100)))
        out.append(("amount", new, f"ask for {cut}% less money ({int(r['amount'])} -> {new} DM)", k))
    k = 1
    while r["duration_months"] - 6 * k >= 6:
        new = int(r["duration_months"] - 6 * k)
        out.append(("duration_months", new, f"pay it back faster ({int(r['duration_months'])} -> {new} months)", k)); k += 1
    if r["savings"] in SAVINGS_ORDER:
        i = SAVINGS_ORDER.index(r["savings"])
        for j in range(i + 1, len(SAVINGS_ORDER)):
            out.append(("savings", SAVINGS_ORDER[j], f"build savings from '{r['savings']}' to '{SAVINGS_ORDER[j]}' DM", j - i))
    if r["checking"] in CHECKING_ORDER:
        i = CHECKING_ORDER.index(r["checking"])
        for j in range(i + 1, len(CHECKING_ORDER)):
            out.append(("checking", CHECKING_ORDER[j], f"bring the checking account from '{r['checking']}' to '{CHECKING_ORDER[j]}' DM", j - i))
    if r["other_debtors"] != "guarantor":
        out.append(("other_debtors", "guarantor", "add a guarantor (someone who promises to pay if you cannot)", 2))
    if r["employment"] in EMPLOY_ORDER[:-1]:
        nxt = EMPLOY_ORDER[EMPLOY_ORDER.index(r["employment"]) + 1]
        out.append(("employment", nxt, f"stay in your job until it counts as '{nxt}' (takes time)", 3))
    return out

def recourse(person, max_options=2):
    """Returns a list of plain-text options that would turn a 'no' into a 'yes'. Empty list = nothing small is enough."""
    person = person[FEATURES_NOAGE]
    if model_noage.predict(person)[0] == 1: return []
    changes = possible_changes(person)
    if not changes: return []
    singles = pd.concat([person] * len(changes), ignore_index=True)
    for i, (f, v, _, _) in enumerate(changes): singles.at[i, f] = v
    ok = model_noage.predict(singles) == 1
    best = {}                                             # smallest working change per detail
    for (f, v, text, effort), good in zip(changes, ok):
        if good and (f not in best or effort < best[f][1]): best[f] = (text, effort)
    if best:
        return [t for t, _ in sorted(best.values(), key=lambda x: x[1])[:max_options]]
    pairs = [(a, b) for a, b in combinations(changes, 2) if a[0] != b[0]]       # no single change works: try two together
    if not pairs: return []
    doubles = pd.concat([person] * len(pairs), ignore_index=True)
    for i, (a, b) in enumerate(pairs): doubles.at[i, a[0]] = a[1]; doubles.at[i, b[0]] = b[1]
    ok2 = model_noage.predict(doubles) == 1
    working = sorted([(a[3] + b[3], f"{a[2]} AND {b[2]}") for (a, b), good in zip(pairs, ok2) if good])
    return [t for _, t in working[:1]]

rejected_ids = df.index[model_noage.predict(df[FEATURES_NOAGE]) == 0]
show = [leyla.index[0]] + [i for i in rejected_ids if i != leyla.index[0]][:2]
for pid in show:
    p = df.loc[[pid]]
    print(f"Applicant #{pid} (approve chance {model_noage.predict_proba(p[FEATURES_NOAGE])[0,1]:.0%}):")
    opts = recourse(p)
    if model_noage.predict(p[FEATURES_NOAGE])[0] == 1: print("   already approved")
    elif opts:
        for o in opts: print("   would get a yes if they:", o)
    else: print("   no small change is enough")

# How many rejected people have a way to a yes? Compare the age groups.
has_way = pd.Series({pid: len(recourse(df.loc[[pid]])) > 0 for pid in rejected_ids})
grp = df.loc[rejected_ids, "young"].map({1: "under 25", 0: "25+"})
print("\nRejected people who have at least one small change that leads to a yes:")
print(pd.DataFrame({"has a way": has_way.groupby(grp).mean().round(3), "rejected people": has_way.groupby(grp).size()}))

**What we see.** About 9 in 10 rejected people have at least one small change that leads to a yes, and there is no difference between the age groups here.

## 11. The explainer

A refused applicant deserves a reason. We read the model's three strongest reasons for each person directly from its weights, then ask a language model (Gemini) to say them in plain words. Gemini does not make the decision and is told to use only those three reasons.

Every answer is saved in `explanations.json` in our repository, so this notebook runs without an API key.

In [ ]:
# The explainer - why did I get this answer?
# What it does: (a) finds the 3 details that pushed THIS person's decision the most (the model's real reasons),
#               (b) asks Gemini to turn them into 3 plain sentences,
#               (c) saves every Gemini answer in explanations.json, so the notebook also runs with no API key.
import json, os, re, time, hashlib, urllib.request

USE_API = False      # False = use the saved Gemini answers (no key needed). To ask Gemini live: run  !pip install -q -U google-genai ,
                     # put your key in Colab Secrets as GEMINI_API_KEY, and set this to True.
CACHE_FILE = "explanations.json"
CACHE_URL = "https://raw.githubusercontent.com/rehim2009book-ui/fairloan/main/explanations.json"
GEMINI_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash"]   # tried in this order

PLAIN = {"checking":"checking account balance", "duration_months":"loan length (months)", "credit_history":"credit history",
         "purpose":"loan purpose", "amount":"loan amount", "savings":"savings", "employment":"time in current job",
         "installment_rate":"share of income going to repayments (code 1-4)", "other_debtors":"co-applicant or guarantor",
         "residence_years":"time at current address (code 1-4)", "property":"most valuable property", "other_plans":"other payment plans",
         "housing":"housing (rent-free / rent / own)", "existing_credits":"number of loans at this bank (code 1-4)", "job":"job type",
         "people_liable":"number of dependants (coded)", "telephone":"registered telephone"}

# ---- (a) the model's real reasons ----
_pre, _clf = model_noage.named_steps["pre"], model_noage.named_steps["model"]
_names = list(_pre.get_feature_names_out())
_avg = np.asarray(_pre.transform(X_train[FEATURES_NOAGE]).mean(axis=0)).ravel() * _clf.coef_[0]   # the average applicant

def _owner(col_name):
    """Which original column does this model input belong to?"""
    if col_name.startswith("num__"): return col_name[5:]
    for c in sorted(FEATURES_NOAGE, key=len, reverse=True):
        if col_name.startswith(f"cat__{c}_"): return c
    return col_name

def top_drivers(person, k=3):
    """Returns the decision, the approve chance, and the k details that pushed hardest toward that decision."""
    xt = _pre.transform(person[FEATURES_NOAGE])
    xt = np.asarray(xt.todense()).ravel() if hasattr(xt, "todense") else np.asarray(xt).ravel()
    push = pd.Series(xt * _clf.coef_[0] - _avg, index=[_owner(n) for n in _names]).groupby(level=0).sum()
    prob = model_noage.predict_proba(person[FEATURES_NOAGE])[0, 1]
    approved = prob >= 0.5
    ranked = push.sort_values(ascending=not approved)          # rejected -> most negative first
    drivers = [{"feature": f, "plain": PLAIN.get(f, f), "value": str(person.iloc[0][f]),
                "effect": "helped" if v > 0 else "hurt", "strength": round(float(v), 2)}
               for f, v in ranked.head(k).items()]
    return approved, prob, drivers

# ---- (b) Gemini, with saved answers ----
if not os.path.exists(CACHE_FILE):                 # fresh runtime: fetch the saved answers from the repo
    try: urllib.request.urlretrieve(CACHE_URL, CACHE_FILE)
    except Exception: pass
try: cache = json.load(open(CACHE_FILE))
except Exception: cache = {}
cache = {k: v for k, v in cache.items() if isinstance(v, dict) and "model" in v}     # ignore answers saved in the old format

_client, _last_call, _warned = None, 0.0, False
def _call_api(prompt):
    """Returns (text, model_name) or (None, None) if the API is not available."""
    global _client, _last_call, _warned
    if not USE_API: return None, None
    try:
        from google import genai
        if _client is None:
            try:
                from google.colab import userdata
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                key = os.environ.get("GEMINI_API_KEY")
            if not key:
                if not _warned: print("No GEMINI_API_KEY found - using saved answers only."); _warned = True
                return None, None
            _client = genai.Client(api_key=key)
        for m in list(GEMINI_MODELS):
            for attempt in range(2):
                time.sleep(max(0, 4.5 - (time.time() - _last_call))); _last_call = time.time()     # stay under the free limit
                try:
                    text = _client.models.generate_content(model=m, contents=prompt).text.strip()
                    GEMINI_MODELS.remove(m); GEMINI_MODELS.insert(0, m)      # remember the one that works
                    return text, m
                except Exception as e:
                    busy = any(w in str(e) for w in ["429", "RESOURCE_EXHAUSTED", "quota", "503"])
                    print(f"  ({m} failed: {str(e)[:80]})")
                    if busy and attempt == 0: time.sleep(30)
                    else: break
    except Exception as e:
        if not _warned: print("Gemini not available:", str(e)[:100]); _warned = True
    return None, None

_used = set()
def gemini(prompt):
    """Saved answer if we have one for exactly this prompt, otherwise ask Gemini and save it."""
    h = hashlib.md5(prompt.encode()).hexdigest()[:12]; _used.add(h)
    if h in cache: return cache[h]["text"], cache[h]["model"]
    text, used = _call_api(prompt)
    if text:
        cache[h] = {"text": text, "model": used}
        json.dump(cache, open(CACHE_FILE, "w"), indent=1)
    return text, used

# How to say each value in normal words (only used when talking to the applicant; the model sees the short labels)
SAY = {
 "credit_history": {"all credits here paid back": "paid back every earlier loan at this bank", "no credits / all paid back": "no earlier loans, or all of them paid back",
                    "existing credits paid so far": "current loans paid on time so far", "critical / credits elsewhere": "a critical account, or loans at other banks",
                    "delays in the past": "late payments in the past"},
 "checking": {"no account": "no checking account at this bank", "<0": "overdrawn (below 0 DM)", "0-200": "between 0 and 200 DM",
              ">=200 or salary": "200 DM or more, or a salary account"},
 "savings": {"none/unknown": "no savings account, or unknown", "<100": "under 100 DM", "100-500": "100 to 500 DM", "500-1000": "500 to 1000 DM", ">=1000": "1000 DM or more"},
 "property": {"none": "no valuable property", "real estate": "real estate (a house, flat or land)", "car/other": "a car or other property"},
 "housing": {"rent-free": "lives rent-free (pays no rent)", "rent": "rents", "own": "owns their home"},
 "other_debtors": {"none": "no co-applicant and no guarantor", "co-applicant": "has a co-applicant", "guarantor": "has a guarantor"},
 "employment": {"<1yr": "less than 1 year", "1-4yr": "1 to 4 years", "4-7yr": "4 to 7 years", ">=7yr": "7 years or more"},
 "telephone": {"no": "no telephone registered", "yes": "has a registered telephone"},
 "other_plans": {"none": "no other payment plans", "bank": "another payment plan at a bank", "stores": "a payment plan at a store"},
}

def build_prompt(approved, drivers):
    lines = "\n".join(f"- {d['plain']}: {SAY.get(d['feature'], {}).get(d['value'], d['value'])} ({d['effect']})" for d in drivers)
    return (f"A bank's computer model {'APPROVED' if approved else 'REJECTED'} a loan application.\n"
            "These are the ONLY reasons the model used, strongest first. 'helped' = counted in the applicant's favour, "
            "'hurt' = counted against the applicant. Money is in Deutsche Mark (DM).\n"
            f"{lines}\n\n"
            "Write exactly 3 short sentences to the applicant ('you'), one per reason, in natural everyday English a 15-year-old understands.\n"
            "Rules: keep every fact the same - same detail, same value, same direction - but say it in normal words, not as a label. "
            "Example: 'savings: 1000 DM or more (helped)' becomes 'Having 1000 DM or more in savings helped you.' "
            "Do not add any other reason, even if one of these sounds surprising. Do not explain why a reason matters. "
            "Do not mention age, sex or nationality. Do not promise anything. No greeting, no sign-off.")

def template_text(approved, drivers):
    parts = ", ".join(f"{d['plain']} ({d['value']})" for d in drivers)
    return f"[no AI answer saved for this case] Your application was {'approved' if approved else 'rejected'}. The details that mattered most were: {parts}."

def explain_person(person):
    """person = one-row table. Returns decision, the model's real reasons, and the plain-language note."""
    approved, prob, drivers = top_drivers(person)
    text, used = gemini(build_prompt(approved, drivers))
    return {"decision": "APPROVED" if approved else "REJECTED", "approve_chance": round(float(prob), 2),
            "drivers": drivers, "text": text or template_text(approved, drivers), "written_by": used or "template"}

def explain(person_id): return explain_person(df.loc[[person_id]])

# ---- try it on Leyla and on one approved person ----
approved_id = int(df.index[model_noage.predict(df[FEATURES_NOAGE]) == 1][0])
for pid in [int(leyla.index[0]), approved_id]:
    r = explain(pid)
    print(f"Applicant #{pid}: {r['decision']} (approve chance {r['approve_chance']:.0%})   [written by: {r['written_by']}]")
    for d in r["drivers"]:
        print(f"   model's real reason: {d['plain']} = {d['value']}  ({d['effect']}, strength {d['strength']})")
    print("   " + r["text"] + "\n")
print(f"Saved answers in {CACHE_FILE}: {len(cache)}")

## 12. The lie check: can we trust the AI's notes?

Two tests on 30 applicants (15 rejected, 15 approved):

- **Test 1, guessing.** Gemini sees the application and the decision but *not* the model's reasons, and must name the three details that mattered most.
- **Test 2, told.** Our tool gives Gemini the model's three real reasons. We check each note: does it mention all three, does it add a reason it was not given, does it mention age, sex or nationality?

The check in Test 2 is simple keyword matching. Any flagged note is printed so a person can read it.

In [ ]:
# The lie check - can we trust the AI's notes?
# What it does, on 30 applicants (15 rejected, 15 approved). Takes about 5 minutes the first time (free API speed limit).
#   Test 1 "guess": Gemini sees the full application and the decision, but NOT the model's reasons, and must name
#                   the 3 details that mattered most. How often does it name the real ones?
#   Test 2 "told":  our tool gives Gemini the model's 3 real reasons. Does the note mention all of them,
#                   add reasons that were not given, or mention age / sex / nationality?
# The word check in Test 2 is simple keyword matching. Every flagged note is printed so a person can read it.
rej = df.index[model_noage.predict(df[FEATURES_NOAGE]) == 0]; app = df.index[model_noage.predict(df[FEATURES_NOAGE]) == 1]
rng = np.random.default_rng(0)
check_ids = list(rng.choice(rej, 15, replace=False)) + list(rng.choice(app, 15, replace=False))

# ---- Test 1: guess ----
def guess_prompt(person, approved):
    r = person.iloc[0]
    lines = "\n".join(f"- {f} ({PLAIN[f]}): {r[f]}" for f in FEATURES_NOAGE)
    return (f"A bank's computer model {'APPROVED' if approved else 'REJECTED'} this loan application:\n{lines}\n\n"
            "Which 3 details mattered MOST for this decision? Answer with exactly 3 names from the list above "
            "(the short name before the bracket), separated by commas. Nothing else.")

guess_scores = []
for pid in check_ids:
    person = df.loc[[pid]]
    approved, _, drivers = top_drivers(person)
    answer, _ = gemini(guess_prompt(person, approved))
    if not answer: continue
    named = {f for f in FEATURES_NOAGE if re.search(rf"\b{f}\b", answer)}
    real = {d["feature"] for d in drivers}
    guess_scores.append(len(named & real) / 3)
if guess_scores:
    print(f"TEST 1 - Gemini GUESSES the reasons ({len(guess_scores)} applicants):")
    print(f"   real reasons it named: {np.mean(guess_scores):.0%}   (picking 3 at random would score {3/len(FEATURES_NOAGE):.0%})")
    print(f"   applicants where it got all 3 right: {np.mean(np.array(guess_scores) == 1):.0%}")
else:
    print("TEST 1 skipped: no Gemini answers available.")

# ---- Test 2: told ----
WORDS = {
 "checking": r"checking|bank account|account balance|current account",
 "duration_months": r"month|loan length|length of the loan|loan term|duration|repayment period",
 "credit_history": r"credit history|(past|previous|earlier|current) (loans?|credits?)|paid back|paid on time|late payments?|paying late|critical|other banks",
 "purpose": r"purpose|other reasons?|another reason|what the loan is for|new car|used car|furniture|radio|\btv\b|television|appliance|repairs?|education|vacation|holiday|retraining|business",
 "amount": r"loan amount|amount of the loan|(?<!savings )amount (you|of money|was|is)|borrow|how much|size of the loan|loan size|loan of \d|asking for",
 "savings": r"\bsavings?\b",
 "employment": r"employ|current job|time in (your )?job|(in|at) (your|the) job",
 "installment_rate": r"income",
 "other_debtors": r"co-?applicant|guarant|co-?signer",
 "residence_years": r"address|residence|lived at",
 "property": r"property|real estate|life insurance|building plan",
 "other_plans": r"payment plans?|installment plans?|other plans?",
 "housing": r"housing|\brent|own(s|ing)? (your|their|a) (own )?home|home ?owner",
 "existing_credits": r"existing (loans|credits)|other loans|number of (loans|credits)",
 "job": r"job type|type of job|kind of job|skilled|management|manager",
 "people_liable": r"dependa?e?nts?",
 "telephone": r"phone",
}
FORBIDDEN = r"\bage\b|\byoung|\bolder?\b|\bsex\b|gender|nationality|foreign"

rows = []
for pid in check_ids:
    r = explain(pid)
    if r["written_by"] == "template": continue
    text = r["text"].lower(); real = [d["feature"] for d in r["drivers"]]
    mentioned = [f for f in real if re.search(WORDS[f], text)]
    extra = [f for f in WORDS if f not in real and re.search(WORDS[f], text)]
    rows.append({"id": pid, "real reasons mentioned": len(mentioned) / 3, "extra": extra,
                 "missing": [f for f in real if f not in mentioned], "forbidden": bool(re.search(FORBIDDEN, text)), "text": r["text"]})
told = pd.DataFrame(rows)
if len(told):
    print(f"\nTEST 2 - Gemini is TOLD the real reasons ({len(told)} applicants):")
    print(f"   real reasons mentioned in the note: {told['real reasons mentioned'].mean():.0%}")
    print(f"   notes that add a reason that was not given: {(told['extra'].str.len() > 0).mean():.0%}")
    print(f"   notes that mention age, sex or nationality: {told['forbidden'].mean():.0%}")
    flagged = told[(told["extra"].str.len() > 0) | (told["missing"].str.len() > 0) | told["forbidden"]]
    print(f"\n   Flagged notes to read by hand: {len(flagged)}")
    for _, row in flagged.iterrows():
        print(f"   #{row['id']}  missing: {row['missing']}  extra: {row['extra']}\n      {row['text']}")
else:
    print("\nTEST 2 skipped: no Gemini answers available.")
print(f"\nSaved answers in {CACHE_FILE}: {len(cache)}")

**What we see.** When Gemini has to guess, it names fewer than half of the model's real reasons, and it almost never gets all three. Asking a chatbot "why did the bank say no?" gives a confident answer that is usually partly wrong. When it is told the real reasons, the notes stick to them.

**Where our own check failed.** The first version of the keyword check flagged 9 of the 30 notes. We read all 9 by hand: every note was correct, and the mistakes were in our check (for example it read "unknown amount of savings" as the loan amount). We fixed those patterns. The check still only looks for words, and it covers 30 people.

## 13. The demo: one applicant in, the full answer out

This puts everything together for one real person from the data: the decision, the model's real reasons, the plain-language note, the twin check, and what would get a yes.

We list all 7 under-25s the Twin Test found and show applicant #470, picked by hand because her reasons are the easiest to follow.

In [ ]:
# The demo - one applicant in, the full answer out
# What it does: puts everything together for one person: decision, the model's real reasons, the plain note,
# the twin check, and what would get a yes. This is what goes in the video.
def twin_check(person):
    """For each age stand-in: if this person has the young-typical value, would the older-typical value change the answer?"""
    out = []; base = model_noage.predict(person[FEATURES_NOAGE])[0]
    for f, yv, ov, kind in pairs:
        if kind != "stand-in for age" or person.iloc[0][f] != yv: continue
        t = person[FEATURES_NOAGE].copy(); t[f] = ov
        if model_noage.predict(t)[0] != base:
            out.append(f"{PLAIN[f]}: '{yv}' -> '{ov}' changes the answer to {'YES' if base == 0 else 'NO'}")
    return out

def check_applicant(person):
    r = explain_person(person)
    lines = [f"DECISION: {r['decision']}  (approve chance {r['approve_chance']:.0%})", "", "WHY (the model's real reasons):"]
    lines += [f"  - {d['plain']} = {d['value']}  ({d['effect']})" for d in r["drivers"]]
    lines += ["", "IN PLAIN WORDS:", "  " + r["text"], "", "TWIN CHECK (details that mostly signal age):"]
    tw = twin_check(person)
    lines += [f"  ! {t}" for t in tw] if tw else ["  ok - no age stand-in changes this answer on its own"]
    if r["decision"] == "REJECTED":
        opts = recourse(person)
        lines += ["", "WHAT WOULD GET A YES:"] + ([f"  - {o}" for o in opts] if opts else ["  - no small change is enough"])
    lines += ["", "Prototype only. Not financial advice."]
    return "\n".join(lines)

# Pick the clearest example for the video: a real under-25 whose answer flips on the age stand-in,
# and where that stand-in is also the model's strongest reason.
try:
    flip_ids = list(cands.index[flipped])
    print(f"Real under-25s whose answer flips when only '{demo_feature}' changes from '{demo_young_val}' to '{demo_old_val}' ({len(flip_ids)}):")
    for pid in flip_ids:
        _, pr, dr = top_drivers(df.loc[[pid]])
        print(f"   #{pid}: approve chance {pr:.0%}, top reasons: " + ", ".join(f"{d['feature']}={d['value']}" for d in dr))
    DEMO_PICK = 470      # picked by hand from the list above: her three reasons are the easiest to follow. All of them are listed.
    best = [pid for pid in flip_ids if top_drivers(df.loc[[pid]])[2][0]["feature"] == demo_feature]
    demo_id = DEMO_PICK if DEMO_PICK in flip_ids else (int(best[0]) if best else int(leyla.index[0]))
except NameError:
    demo_id = int(leyla.index[0])

print(f"\n================ DEMO: applicant #{demo_id} ================")
print(check_applicant(df.loc[[demo_id]]))

# keep only the saved answers this notebook actually uses
cache = {h: cache[h] for h in sorted(_used) if h in cache}
json.dump(cache, open(CACHE_FILE, "w"), indent=1)
print(f"\nSaved Gemini answers used by this notebook: {len(cache)}")

RUN_APP = False      # set True to open a small web form (needs:  !pip install -q gradio)
if RUN_APP:
    import gradio as gr
    base_person = df.loc[[demo_id]].copy()
    def form(housing, employment, savings, checking, amount, duration_months, other_debtors):
        p = base_person.copy()
        for k, v in dict(housing=housing, employment=employment, savings=savings, checking=checking,
                         amount=int(amount), duration_months=int(duration_months), other_debtors=other_debtors).items():
            p[k] = v
        return check_applicant(p)
    b = base_person.iloc[0]
    demo = gr.Interface(fn=form, title="TwinCheck - loan decision, explained and checked",
        inputs=[gr.Dropdown(["rent-free", "rent", "own"], value=b["housing"], label="Housing"),
                gr.Dropdown(EMPLOY_ORDER, value=b["employment"], label="Time in current job"),
                gr.Dropdown(SAVINGS_ORDER, value=b["savings"], label="Savings (DM)"),
                gr.Dropdown(["no account"] + CHECKING_ORDER, value=b["checking"], label="Checking account (DM)"),
                gr.Number(value=int(b["amount"]), label="Loan amount (DM)"),
                gr.Slider(6, 72, value=int(b["duration_months"]), step=6, label="Loan length (months)"),
                gr.Dropdown(["none", "co-applicant", "guarantor"], value=b["other_debtors"], label="Co-applicant or guarantor")],
        outputs=gr.Textbox(label="TwinCheck says", lines=22))
    demo.launch(share=True)

## 14. Results summary

| | Result |
|---|---|
| Baseline, approve everyone (test set) | accuracy 70.0%, cost 300 |
| Our model (test set) | accuracy 67.0%, catches 68.3% of bad risks, **cost 142** |
| Approval gap, all 1,000 people | 17.4 points with age, **13.8 points with age deleted** |
| Age guesser, without seeing age | scores 0.78 (0.50 = coin flip) |
| Twin Test, housing only | decision flips for 13% of applicants; 7 of 69 young rent-free applicants |
| Young good payers wrongly rejected | 37.5%, against 26.8% for older good payers |
| Fix A, delete stand-in details | cost 617 → 603, young good payers wrongly rejected 37.5% → 36.4% |
| Fix B, lower pass mark for under-25s | cost 617 → 708, young good payers wrongly rejected 37.5% → 27.3% |
| Rejected people with a way to a yes | about 9 in 10, both age groups |
| Gemini guessing the model's reasons | names 42% of them (random = 18%) |
| Gemini told the model's reasons | all 30 notes gave the three real reasons and added none |

**Main takeaway:** deleting the age column does not remove age bias, because other details give age away. The bias we found is modest, it can be measured, and fixing it has a price.

## 15. Where it fails, and limits

- **Small groups.** Only 149 of 1,000 applicants are under 25, and only 88 of them are good payers. Our age results are strong signs, not proof.
- **Old data.** The loans are from 1970s Germany, in Deutsche Mark. The data only contains loans that were granted, and bad risks were over-sampled. Today's banks may look very different.
- **No free fix.** Fix A barely helps. Fix B costs the bank about 15% more, uses age on purpose, and tips the approval gap the other way.
- **A rule we cannot explain.** The model learned that owning real estate *hurts* an application. We checked the label and it is correct. The explainer shows this honestly, but it would confuse an applicant.
- **Fairness has more than one meaning.** We measured approval gaps and wrongful rejections. Making one equal can make another unequal.
- **The lie check is small and simple:** 30 people and keyword matching. Our first version of it was wrong about 9 notes.
- **"What would get a yes" ignores real life.** Saving 500 DM is not equally easy for everyone.

## 16. What we would do next

1. Add reliability ranges to every gap by re-sampling the data many times, because the young group is small.
2. Run the same audit on a modern lending dataset, and turn TwinCheck into one function that can audit any bank's model.
3. Ask real young applicants whether the explanation and the suggested change actually help them.

## Sources

- **Data:** Hofmann, H. (1994). Statlog (German Credit Data). UCI Machine Learning Repository. https://doi.org/10.24432/C5NC77 (licence CC BY 4.0)
- **Corrected code table:** Grömping, U. (2019). South German Credit Data: Correcting a Widely Used Data Set. http://www1.beuth-hochschule.de/FB_II/reports/Report-2019-004.pdf
- **Ideas we built on:** paired testing of software for discrimination, Galhotra, Brun and Meliou (2017), https://arxiv.org/pdf/1709.03221 ; checking AI explanations against a model, https://arxiv.org/html/2412.10220v1
- **Background:** Amnesty International (2021), "Xenophobic Machines", https://www.amnesty.org/en/latest/news/2021/10/xenophobic-machines-dutch-child-benefit-scandal/ ; EU AI Act, Annex III, https://artificialintelligenceact.eu/annex/3/
- **Libraries:** pandas, NumPy, scikit-learn, matplotlib, ucimlrepo; google-genai and Gradio are optional.
- **AI tools:** Gemini API (`gemini-3.5-flash-lite`) writes the plain-language explanations. We used Claude to help plan the project, research the background, draft and debug the code, and run the notebook. We read and tested every part, and we changed the plan when the results did not match what we expected.

*This project is a prototype and is not financial advice.*